In [93]:
import pandas as pd
from goatools.obo_parser import GODag
from goatools.mapslim import mapslim
from collections import Counter

In [2]:
ortho = pd.read_csv('ORTHOLOGY-ALLIANCE_COMBINED.tsv', sep='\t', comment='#')

In [5]:
worm_human_ortho = ortho[(ortho.Gene1SpeciesName=='Caenorhabditis elegans') & (ortho.Gene2SpeciesName=='Homo sapiens')]
worm_human_genes = set(worm_human_ortho.Gene1Symbol)

In [259]:
disease = pd.read_csv('DISEASE-ALLIANCE_WB.tsv', sep='\t', comment='#', low_memory=False)
worm_disease = set(disease[disease.DBobjectType=='gene'].DBObjectSymbol)

# f_out=open("worm_disease.gene.tsv", "w")
# for g in worm_disease:
#     print(g, file=f_out)
# f_out.close()

In [252]:
#godag = GODag("go-basic.obo", optional_attrs=['relationship'])
godag = GODag("gene_ontology.WS298.obo", optional_attrs=['relationship'])
goslim_dag = GODag("goslim_generic.obo", optional_attrs=['relationship'])
goslim_agr_dag = GODag("goslim_agr.obo") 
worm_go = pd.read_csv('gene_association.WS298.wb', sep='\t', comment='!', header=None, low_memory=False)

###find replacements:
replaced_by = {}
current_id = None
with open("go-basic.obo") as f:
    for line in f:
        line = line.strip()
        if line.startswith("id: GO:"):
            current_id = line.split("id: ")[1]
        elif line.startswith("replaced_by:"):
            replaced_by[current_id] = line.split("replaced_by: ")[1].strip()

def map_to_slim(go_id):
    go_id = replaced_by.get(go_id, go_id)  # substitute if obsolete-with-replacement
    try:
        direct_ancestors, all_ancestors = mapslim(go_id, godag, custom_slim_dag)
        return direct_ancestors
    except:
        return set([])
    
def retrieve_go_name(go_id):
    try:
        return godag[go_id].name
    except KeyError:
        return None

gene_ontology.WS298.obo: fmt(1.2) rel(2025-07-22) 43,230 Terms; optional_attrs(relationship)
goslim_generic.obo: fmt(1.2) rel(go/2026-06-15/subsets/goslim_generic.owl) 206 Terms; optional_attrs(relationship)
goslim_agr.obo: fmt(1.2) rel(go/2026-06-15/subsets/goslim_agr.owl) 96 Terms


In [248]:
###build a custom set of ids
custom_slim_ids = set(t.id for t in goslim_dag.values() if not t.is_obsolete) | \
                   set(t.id for t in goslim_agr_dag.values() if not t.is_obsolete)
extra_terms = [
    "GO:0016020",  # membrane
    "GO:0005737",  # cytoplasm
    "GO:0043005",  # neuron projection  (covers axon/dendrite via is_a)
    "GO:0045202",  # synapse
    "GO:0000228",  # (already in generic, just confirming)
    "GO:0030054",  # cell junction (covers gap junction via is_a)
    "GO:0005575",  # root -- decide: keep for "unlocalized" bucket, or exclude
]

for go_id in extra_terms:
    if go_id in godag:
        custom_slim_ids.add(go_id)

print("combined generic+agr: {} Terms".format(len(custom_slim_ids)))

custom_slim_dag = GODag("goslim_generic.obo", optional_attrs=['relationship'])  # start from generic
for go_id in custom_slim_ids:
    if go_id not in custom_slim_dag and go_id in godag:
        custom_slim_dag[go_id] = godag[go_id]

#hard fix the 5615 obsolete annotation.
custom_slim_dag["GO:0005615"] = godag.get("GO:0005576")  # borrow the still-valid extracellular region term

print(len(custom_slim_dag))

combined generic+agr: 169 Terms
goslim_generic.obo: fmt(1.2) rel(go/2026-06-15/subsets/goslim_generic.owl) 206 Terms; optional_attrs(relationship)
236


In [260]:
###MAIN LOOP TO EXTRACT GO TERMS
rows = []
for g in worm_disease:
    go_term_cc = worm_go[
        (worm_go.loc[:,2]==g) &
        (worm_go.loc[:,3].isin(['located_in', 'part_of']))
    ].loc[:,4]

    if len(go_term_cc) == 0:
        rows.append({'gene': g, 'go_term': None, 'go_slim_term': None, 'cellular_component': None})
        continue

    for t in go_term_cc:
        slim_term = map_to_slim(t)
        if len(slim_term) == 0:
            rows.append({'gene': g, 'go_term': t, 'go_slim_term': None, 'cellular_component': None})
        else:
            st = slim_term.pop()
            rows.append({
                'gene': g,
                'go_term': t,
                'go_slim_term': st,
                'cellular_component': retrieve_go_name(st)
            })

worm_disease_go = pd.DataFrame(rows, columns=['gene', 'go_term', 'go_slim_term', 'cellular_component'])

worm_disease_go_unique = (
    worm_disease_go
    .groupby(['gene', 'go_slim_term', 'cellular_component'], as_index=False, dropna=False)
    .agg(go_terms=('go_term', lambda x: sorted(set(v for v in x if v is not None))))
)

In [261]:
Counter(worm_disease_go_unique.cellular_component)

Counter({'membrane': 1697,
         'cytoplasm': 1328,
         'protein-containing complex': 1233,
         'nucleus': 1095,
         'cellular_component': 896,
         'plasma membrane': 816,
         'mitochondrion': 434,
         nan: 414,
         'cytosol': 325,
         'neuron projection': 267,
         'endoplasmic reticulum': 253,
         'organelle': 241,
         'cytoskeleton': 190,
         'extracellular region': 188,
         'cell projection': 155,
         'synapse': 155,
         'Golgi apparatus': 148,
         'cytoplasmic vesicle': 136,
         'chromosome': 109,
         'cell junction': 106,
         'microtubule organizing center': 87,
         'cilium': 84,
         'lysosome': 83,
         'endosome': 80,
         'ribosome': 69,
         'nucleolus': 59,
         'nucleoplasm': 59,
         'extracellular matrix': 48,
         'nuclear envelope': 45,
         'peroxisome': 43,
         'vacuole': 30,
         'nuclear chromosome': 15,
         'lipid drop

In [262]:
absent_check = []
for g in worm_disease:
    n_total = len(worm_go[worm_go.loc[:,2]==g])
    absent_check.append({'gene': g, 'n_total_rows': n_total})

absent_df = pd.DataFrame(absent_check)
absent_df['n_total_rows'].eq(0).sum()  # total genes fully absent from GAF

fully_absent = absent_df[absent_df['n_total_rows']==0]['gene']

for g in fully_absent:
    matches = worm_go[worm_go.loc[:,2].str.contains(g, case=False, na=False)].loc[:,2].unique()
    print(g, "->", matches[:5] if len(matches) > 0 else "NO MATCH AT ALL")

frm-5.2 -> NO MATCH AT ALL
mir-1 -> NO MATCH AT ALL
Y37A1A.4 -> NO MATCH AT ALL
frm-8 -> NO MATCH AT ALL
R166.3 -> NO MATCH AT ALL
C18E9.7 -> NO MATCH AT ALL
F56A8.9 -> NO MATCH AT ALL
gipc-1 -> NO MATCH AT ALL
F17H10.1 -> NO MATCH AT ALL
bah-3 -> NO MATCH AT ALL
poml-3 -> NO MATCH AT ALL
K09E4.4 -> NO MATCH AT ALL
mir-84 -> NO MATCH AT ALL
elk-2 -> NO MATCH AT ALL
wac-1.2 -> NO MATCH AT ALL
bgnt-1.6 -> NO MATCH AT ALL
wdr-37 -> NO MATCH AT ALL
let-7 -> ['let-70' 'let-711' 'let-716' 'let-721' 'let-754']
mir-77 -> NO MATCH AT ALL
hsp-16.1 -> ['hsp-16.11']
bgnt-1.7 -> NO MATCH AT ALL
bgnt-1.2 -> NO MATCH AT ALL
K09E2.1 -> NO MATCH AT ALL
bgnt-1.3 -> NO MATCH AT ALL
K08B4.7 -> NO MATCH AT ALL
W01C8.5 -> NO MATCH AT ALL
trf-2 -> NO MATCH AT ALL
elmd-1 -> NO MATCH AT ALL
hsp-16.48 -> NO MATCH AT ALL
rpms-1 -> NO MATCH AT ALL
F38A5.2 -> ['F38A5.22']
gcy-2 -> ['gcy-20' 'gcy-21' 'gcy-22' 'gcy-23' 'gcy-25']
C29F5.8 -> NO MATCH AT ALL
mdmh-35 -> NO MATCH AT ALL
dip-2 -> NO MATCH AT ALL
reps-1 ->